# Part 2 · Agent Patterns and Harness Engineering

<font color='red'>FDE Crash Course · Episode 1 · Part 2</font>

In Part 1, Gemini learned to **use tools**. One question, a tool or two, one answer.

Real client problems are bigger. "Read this ticket, send it to the right team, draft a reply, and let a human approve it before it goes out." That's many steps, and the order matters.

Engineers keep solving these problems with the same few **patterns**. In this notebook you'll build 7 of them with the **OpenAI Agents SDK**, a small Python framework for agents made by OpenAI. It's made by OpenAI, but it works with **any model**. We'll keep using Gemini.

Then we put the patterns together into a **harness**: everything around the model that makes an agent reliable enough for a client.

By the end, when a client describes a problem, you'll know which pattern fits, and how to design the harness around it.


| Pattern | In OpenAI Agents SDK | Use it when | FreshKart example |
|---|---|---|---|
| Prompt chaining | `Runner.run` one after another | Fixed steps, one after another | Meeting notes → email |
| Router | `handoffs=[...]` | Different inputs need different handling | Ticket → right team |
| Parallel calls | `asyncio.gather` | Independent jobs on the same input | Review → sentiment + issues + reply |
| ReAct agent | `Agent` with `tools` | Steps aren't known in advance | Travel questions with live tools |
| Orchestrator–worker | `agent.as_tool()` | Task splits into parts that need different experts | Proposal from 3 specialists |
| Reflection | `output_type` + a `for` loop | Clear rules for "good" | Cold email that follows the rules |
| Human in the loop | `@function_tool(needs_approval=True)` | Risky or customer-facing actions | Approve reply before sending |

**These patterns combine.** A real FreshKart bot is a **router** that sends "where is my order" tickets to a **ReAct agent**, with a **human in the loop** before any refund. Let's build exactly that.

## 1. Setup

- `openai-agents` is the SDK. Its Python name is just `agents`
- `[litellm]` adds **LiteLLM**, a helper that lets the SDK talk to non-OpenAI models like Gemini

In [9]:
!pip install "openai-agents[litellm]" -q

### API key

Same key as Part 1. If you haven't added it yet:

1. Get your free Gemini API key from https://aistudio.google.com
2. In Colab, click the 🔑 **key icon** on the left
3. Add a secret named `GEMINI_API_KEY` and turn on **Notebook access**

LiteLLM reads the key from an environment variable called `GEMINI_API_KEY`, so we copy it there.

In [2]:
from IPython.display import Markdown, display
from agents import set_tracing_disabled
from google.colab import userdata
import os

# Read the key from Colab Secrets (never typed in the code)
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

# Tracing sends run logs to OpenAI's dashboard, which needs an OpenAI key. We don't have one, so turn it off.
set_tracing_disabled(True)

MODEL = "litellm/gemini/gemini-3.8-flash"

# Confirm the key loaded (shows only the first 6 characters)
print("✅ Gemini key loaded:", os.environ["GEMINI_API_KEY"][:6], "... | Model:", MODEL)

✅ Gemini key loaded: AQ.Ab8 ... | Model: litellm/gemini/gemini-3.8-flash


The model name has 3 parts:

```
litellm / gemini / gemini-3.8-flash
   |        |            |
   |        |            +-- the model
   |        +-- the provider
   +-- "send this through LiteLLM"
```

**Have an OpenAI key?** Set `OPENAI_API_KEY` and use an OpenAI model name like `MODEL = "gpt-5.5"`. Nothing else in this notebook changes.

Open AI

In [ ]:
from IPython.display import Markdown, display
from agents import set_tracing_disabled
from google.colab import userdata
import os

# Read the key from Colab Secrets (never typed in the code)
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# With an OpenAI key, tracing works, so you can view runs on OpenAI's dashboard
set_tracing_disabled(False)

MODEL = "gpt-6.1-sol"   # OpenAI models need no "litellm/" prefix

# Confirm the key loaded (shows only the first 6 characters)
print("✅ OpenAI key loaded:", os.environ["OPENAI_API_KEY"][:6], "... | Model:", MODEL)

✅ OpenAI key loaded: sk-pro ... | Model: gpt-6.1-sol


Claud Antropic


In [ ]:
from IPython.display import Markdown, display
from agents import set_tracing_disabled
from google.colab import userdata
import os

# Read the key from Colab Secrets (never typed in the code)
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

# No OpenAI key, so turn tracing off
set_tracing_disabled(True)

MODEL = "litellm/anthropic/claude-sonnet-5-5"

# Confirm the key loaded (shows only the first 6 characters)
print("✅ Anthropic key loaded:", os.environ["ANTHROPIC_API_KEY"][:6], "... | Model:", MODEL)

✅ Anthropic key loaded: sk-ant ... | Model: litellm/anthropic/claude-sonnet-5-5


Groq

In [ ]:
from IPython.display import Markdown, display
from agents import set_tracing_disabled
from google.colab import userdata
import os

# Read the key from Colab Secrets (never typed in the code)
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# No OpenAI key, so turn tracing off
set_tracing_disabled(True)

MODEL = "litellm/groq/openai/gpt-oss-120b"

# Confirm the key loaded (shows only the first 6 characters)
print("✅ Groq key loaded:", os.environ["GROQ_API_KEY"][:6], "... | Model:", MODEL)

✅ Groq key loaded: gsk_rC ... | Model: litellm/groq/openai/gpt-oss-120b


## 2. The Gemini SDK in 2 Minutes-

The SDK has very few pieces. That's its whole idea.

| Piece | What it is | Used in |
|---|---|---|
| `Agent` | A model + instructions (+ tools) | Every pattern |
| `Runner` | Runs an agent and gives back the result | Every pattern |
| `@function_tool` | Turns a Python function into a tool | ReAct, Human in the loop |
| `handoffs` | An agent passes the conversation to another agent | Router |
| `agent.as_tool()` | An agent used as a tool by another agent | Orchestrator–worker |

There's no `Workflow` class. When **you** decide the steps, you just write normal Python.

Let's make our first agent.

## 2. The SDK in 2 Minutes, OpenAI

The SDK has very few pieces. That's its whole idea.

| Piece | What it is | Used in |
|---|---|---|
| `Agent` | A model + instructions (+ tools) | Every pattern |
| `Runner` | Runs an agent and gives back the result | Every pattern |
| `@function_tool` | Turns a Python function into a tool | ReAct, Human in the loop |
| `handoffs` | An agent passes the conversation to another agent | Router |
| `agent.as_tool()` | An agent used as a tool by another agent | Orchestrator–worker |

There's no `Workflow` class. When **you** decide the steps, you just write normal Python.

**OpenAI notes**
- `MODEL = "gpt-6.1-sol"`: no `litellm/` prefix needed, because OpenAI models are supported natively
- Tracing can stay **on**: view every run on OpenAI's dashboard (platform.openai.com → Traces)

Let's make our first agent.


## 2. The SDK in 2 Minutes, Claude

The SDK has very few pieces. That's its whole idea.

| Piece | What it is | Used in |
|---|---|---|
| `Agent` | A model + instructions (+ tools) | Every pattern |
| `Runner` | Runs an agent and gives back the result | Every pattern |
| `@function_tool` | Turns a Python function into a tool | ReAct, Human in the loop |
| `handoffs` | An agent passes the conversation to another agent | Router |
| `agent.as_tool()` | An agent used as a tool by another agent | Orchestrator–worker |

There's no `Workflow` class. When **you** decide the steps, you just write normal Python.

**Claude notes**
- `MODEL = "litellm/anthropic/claude-sonnet-5-5"`: runs through **LiteLLM**
- Tracing **off**: it needs an OpenAI key
- Don't set `temperature` in model settings: Claude's API no longer accepts it

Let's make our first agent.



## 2. The SDK in 2 Minutes, Groq

The SDK has very few pieces. That's its whole idea.

| Piece | What it is | Used in |
|---|---|---|
| `Agent` | A model + instructions (+ tools) | Every pattern |
| `Runner` | Runs an agent and gives back the result | Every pattern |
| `@function_tool` | Turns a Python function into a tool | ReAct, Human in the loop |
| `handoffs` | An agent passes the conversation to another agent | Router |
| `agent.as_tool()` | An agent used as a tool by another agent | Orchestrator–worker |

There's no `Workflow` class. When **you** decide the steps, you just write normal Python.

**Groq notes**
- `MODEL = "litellm/groq/openai/gpt-oss-120b"`: runs through **LiteLLM**
- Tracing **off**: it needs an OpenAI key
- Use a chat model that supports tools (`gpt-oss-120b` or `gpt-oss-20b`), not guard or whisper models
- Free tier and very fast, but has rate limits: if you get `429`, wait a minute

Let's make our first agent.

In [10]:
from agents import Agent, Runner

explainer = Agent(
    name="Explainer",
    model=MODEL,
    instructions="You explain tech ideas in simple words, in 2 lines max.",
)

result = await Runner.run(explainer, "What is an AI agent?")

print(result.final_output)

An AI agent is a smart program that can make decisions and take actions on its own to reach a specific goal. 
Unlike a regular chatbot, it doesn't just talk—it actively gets tasks done for you from start to finish.


- `instructions` is the system prompt: **how** the agent should behave.
- `Runner.run()` sends the user prompt and returns a result. The answer is in `.final_output`.
- `await` waits for the reply. The SDK is **async** (like MCP in Part 1), and Colab lets you use `await` directly in a cell.

**Outside a notebook** (a normal `.py` file), use `Runner.run_sync(explainer, "...")` instead. No `await` needed.

In [5]:
## Additional
# !pip install -q -U "openai-agents[litellm]"

In [6]:
# from agents import Agent, Runner, set_tracing_disabled
# from google.colab import userdata
# import os

# os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")       # key from Colab Secrets
# set_tracing_disabled(True)                                          # no OpenAI key, so tracing off
# MODEL = "litellm/gemini/gemini-3.8-flash"
# print("✅ Gemini key loaded:", os.environ["GEMINI_API_KEY"][:6], "... | Model:", MODEL)

# coach = Agent(
#     name="Coach",
#     instructions="You are a friendly fitness coach. Answer in 3 short bullets.",
#     model=MODEL,
# )

# result = await Runner.run(coach, "What should I eat before a workout?")
# print(result.final_output)

In [7]:
# from agents import Agent, Runner, set_tracing_disabled
# from google.colab import userdata
# import os

# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")       # key from Colab Secrets
# set_tracing_disabled(False)                                         # tracing works with an OpenAI key
# MODEL = "gpt-6.1-sol"                                               # native: no "litellm/" prefix
# print("✅ OpenAI key loaded:", os.environ["OPENAI_API_KEY"][:6], "... | Model:", MODEL)

# coach = Agent(
#     name="Coach",
#     instructions="You are a friendly fitness coach. Answer in 3 short bullets.",
#     model=MODEL,
# )

# result = await Runner.run(coach, "What should I eat before a workout?")
# print(result.final_output)

In [ ]:
# from agents import Agent, Runner, set_tracing_disabled
# from google.colab import userdata
# import os

# os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY") # key from Colab Secrets
# set_tracing_disabled(True)                                          # no OpenAI key, so tracing off
# MODEL = "litellm/anthropic/claude-sonnet-5-5"
# print("✅ Anthropic key loaded:", os.environ["ANTHROPIC_API_KEY"][:6], "... | Model:", MODEL)

# coach = Agent(
#     name="Coach",
#     instructions="You are a friendly fitness coach. Answer in 3 short bullets.",
#     model=MODEL,                                                    # no temperature setting for Claude
# )

# result = await Runner.run(coach, "What should I eat before a workout?")
# print(result.final_output)

In [ ]:
# from agents import Agent, Runner, set_tracing_disabled
# from google.colab import userdata
# import os

# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")           # key from Colab Secrets
# set_tracing_disabled(True)                                          # no OpenAI key, so tracing off
# MODEL = "litellm/groq/openai/gpt-oss-120b"                          # a Groq model that supports tools
# print("✅ Groq key loaded:", os.environ["GROQ_API_KEY"][:6], "... | Model:", MODEL)

# coach = Agent(
#     name="Coach",
#     instructions="You are a friendly fitness coach. Answer in 3 short bullets.",
#     model=MODEL,
# )

# result = await Runner.run(coach, "What should I eat before a workout?")
# print(result.final_output)

What each part does:

os.environ[...] moves the key from 🔑 Secrets to where LiteLLM looks for it.
Agent(...) is the worker: a name, a job description and a model.
await Runner.run(...) gives the agent the task and waits for the answer.
result.final_output is the agent's final reply.

If something fails:

ImportError: restart the session after the pip install.
SecretNotFoundError: check the secret's name in 🔑 and that Notebook access is on.
RuntimeError: event loop: use await Runner.run(...) as shown, not Runner.run_sync(...).

## 3. Workflows vs Agents

There are 2 ways to build with LLMs:

| | Workflow | Agent |
|---|---|---|
| Who decides the steps? | **You**, in code | **The LLM**, while it runs |
| Path | Same every time | Changes for each input |
| Easy to test? | Yes | Harder |
| Good for | Tasks you can draw on a whiteboard | Open-ended tasks |

<img src="https://raw.githubusercontent.com/pratik-gond/FDE_images/main/workflow_vs_agent.png" width="600">

The 7 patterns sit on this line:

| Pattern | Who decides? |
|---|---|
| 1. Prompt chaining | You (workflow) |
| 2. Router | The LLM picks **one** route |
| 3. Parallel calls | You (workflow) |
| 4. ReAct agent | The LLM decides every step |
| 5. Orchestrator–worker | The LLM decides every step |
| 6. Reflection | Add to either |
| 7. Human in the loop | Add to either |

**FDE rule:** start with the simplest pattern that works. Clients care that it's reliable, not that it's "agentic".

## 4. Pattern 1: Prompt Chaining

**The idea:** break a big task into small steps. Each step does one job and passes its output to the next.

```
notes  -->  Summarise  -->  Write email  -->  Polish  -->  final email
```

**FreshKart task:** after a client call, turn messy meeting notes into a follow-up email.

Why not one big prompt? Small steps are **easier to test and fix**. If the email is wrong, you can see exactly which step went wrong.

First, one agent per step:

In [11]:
summariser = Agent(
    name="Summariser",
    model=MODEL,
    instructions="Summarise client meeting notes in 3 bullet points. Then list the action items as: who - what - by when.",
)

email_writer = Agent(
    name="Email Writer",
    model=MODEL,
    instructions="Write a short, friendly follow-up email to the client from this meeting summary. Include the action items.",
)

polisher = Agent(
    name="Polisher",
    model=MODEL,
    instructions="Polish this email: add a subject line, keep it under 150 words, warm and professional tone. Return only the final email.",
)

Now chain them. No special class: run one agent, pass its `final_output` to the next.

In [12]:
notes = """
call w/ Priya (CTO, FreshKart) + Arjun (ops head). 45 min.
- support team gets ~2000 tickets/day, 60% are "where is my order"
- want AI to answer those automatically, rest go to humans
- they use Freshdesk. Priya to share API access by Friday
- we send a demo by 15th. Arjun worried about wrong refund promises
- budget approval needs their CEO, meeting next month
"""

summary = await Runner.run(summariser, notes)  # notes → summary
email = await Runner.run(email_writer, summary.final_output)  # summary → draft email
final = await Runner.run(polisher, email.final_output) # draft → final email

print(final.final_output)

Subject: Great speaking today | Next steps for FreshKart

Hi Priya,

Thank you for taking the time to connect today! It was wonderful learning more about FreshKart's goals to automate order tracking while keeping strict guardrails around refunds. 

To keep our momentum going ahead of your CEO budget meeting next month, here are our agreed-upon next steps:

* **Priya:** Share Freshdesk API access by **Friday**
* **Our team:** Deliver the tailored product demo by the **15th**

Please let me know if any questions come up in the meantime. I'm really looking forward to getting the demo into your hands soon!

Warm regards,

[Your Name]  
[Your Title]


### See every step

Each step is its own variable, so debugging is just printing:

In [13]:
steps = {"Summarise": summary, "Write email": email, "Polish": final}

for name, step in steps.items():
    print("=====", name, "=====")
    print(step.final_output)
    print()

===== Summarise =====
**Meeting Summary:**

* FreshKart aims to automate "where is my order" inquiries (60% of their 2,000 daily Freshdesk tickets) using AI, while routing other issues to human agents.
* Operations raised a key concern regarding guardrails to ensure the AI does not make incorrect refund commitments.
* Commercial progress depends on delivering a demo ahead of a CEO budget approval meeting scheduled for next month.

**Action Items:**

* Priya - Share Freshdesk API access - by Friday
* Our team - Send product demo - by the 15th

===== Write email =====
Subject: Great speaking today! Next steps for FreshKart

Hi Priya,

Thank you for taking the time to chat today! It was great learning more about FreshKart's goals to automate order tracking inquiries while ensuring strict guardrails are in place around refunds. 

As discussed, here are the next steps to keep things moving ahead of your CEO budget meeting next month:

**Action Items:**
* **Priya:** Share Freshdesk API acces

Because it's plain Python, adding a check between steps is just an `if`. For example: stop the chain if the summary has no action items.

**Use it when:** the task has clear, fixed steps. Reports, emails, document processing.

**Try it:** add a 4th step with a `translator` agent that turns the email into Hindi.

## 5. Pattern 2: Router

**The idea:** one LLM looks at the input and decides **where it should go**. Each route has its own specialist.

```
                  +-->  billing agent
ticket --> triage +-->  technical agent
                  +-->  general agent
```

**FreshKart task:** send each support ticket to the right team.

The SDK has a built-in feature for this: **handoffs**. A triage agent reads the ticket and **hands the conversation over** to one specialist. The specialist then answers the customer directly.

**Cost trick:** picking a team is an easy job, so the triage agent runs on a smaller, cheaper model (`ROUTE_MODEL`).

First, the 3 specialists. `handoff_description` is a one-line job title that tells the triage agent **when** to pick this one.

In [14]:
billing_agent = Agent(
    name="Billing Support",
    model=MODEL,
    handoff_description="Payments, double charges, refunds and coupons",
    instructions="You are FreshKart's billing support. Reply politely in 2 lines. Never promise a refund amount.",
)

technical_agent = Agent(
    name="Tech Support",
    model=MODEL,
    handoff_description="App crashes, login problems and bugs",
    instructions="You are FreshKart's tech support. Give simple fix steps in 3 bullet points.",
)

general_agent = Agent(
    name="General Support",
    model=MODEL,
    handoff_description="Everything else: delivery areas, store hours, general questions",
    instructions="You are FreshKart's friendly support agent. Reply in 2 lines.",
)

Now the triage agent. It runs on the cheap model, and its `handoffs` list is the 3 specialists.

In [15]:
ROUTE_MODEL = "litellm/gemini/gemini-3.5-flash-lite"

triage_agent = Agent(
    name="Triage",
    model=ROUTE_MODEL,
    instructions="Decide which FreshKart team should handle this support ticket, and hand off to them. Don't answer yourself.",
    handoffs=[billing_agent, technical_agent, general_agent],
)

In [16]:
tickets = [
    "I was charged twice for my order yesterday!",
    "The app crashes every time I open my cart.",
    "Do you deliver to Whitefield, Bangalore?",
    "I have paid extra for order"
]

for ticket in tickets:
    result = await Runner.run(triage_agent, ticket)
    print("Router picked:", result.last_agent.name)
    print("Reply:", result.final_output)
    print()

Router picked: Billing Support
Reply: I sincerely apologize for the inconvenience caused by the duplicate charge on your order. 
Please share your order ID with us, and our team will promptly investigate the transaction to assist you.

Router picked: Tech Support
Reply: Here are 3 quick steps to fix the cart crash:

* **Update the app:** Open the Google Play Store or Apple App Store, search for **FreshKart**, and tap **Update** if a newer version is available.
* **Clear app cache:** Go to your phone’s **Settings > Apps > FreshKart > Storage** and tap **Clear Cache** (if you're on iPhone, close the app completely from the app switcher).
* **Reinstall the app:** Delete the FreshKart app, restart your phone, and reinstall it from the app store to clear any corrupted cart data.

Router picked: General Support
Reply: Yes, we deliver across Whitefield, Bangalore! You can enter your specific pincode on our app or website to view available delivery slots.

Router picked: Billing Support
Reply:

`result.last_agent` is the agent that gave the final answer, so you can always see **where** each ticket went.

Behind the scenes, a handoff is just a tool. The triage agent gets tools like `transfer_to_billing_support` and the LLM picks one.

**Use it when:** different kinds of input need different handling. You can also send easy questions to a cheap model and hard ones to a big model.

**Try it:** add a 4th team, `delivery`, for "where is my order" tickets.

## 6. Pattern 3: Parallel Calls

**The idea:** when steps **don't depend on each other**, run them at the same time, then combine the results.

```
         +-->  Sentiment  --+
review --+-->  Issues     --+-->  Combine  -->  report
         +-->  Reply      --+
```

**FreshKart task:** for each customer review, find the sentiment, list the issues and draft a reply, all at once.

In [17]:
sentiment_agent = Agent(
    name="Sentiment",
    model=MODEL,
    instructions="Is this review positive, negative or mixed? Answer in one word.",
)

issues_agent = Agent(
    name="Issues",
    model=MODEL,
    instructions="List the product or service problems in this review as short bullet points.",
)

reply_agent = Agent(
    name="Reply",
    model=MODEL,
    instructions="Write a 2-line reply from FreshKart to this review.",
)

Again, no special class. Python's `asyncio.gather` runs several `Runner.run` calls **at the same time** and waits for all of them.

In [18]:
import asyncio
import time

review = "Fast delivery as always, but the milk was already expired and the coupon didn't apply at checkout. Support chat took 40 minutes."

start = time.time()

sentiment, issues, reply = await asyncio.gather(
    Runner.run(sentiment_agent, review),
    Runner.run(issues_agent, review),
    Runner.run(reply_agent, review),
)

print(f"Took {time.time() - start:.1f} seconds")

Took 5.3 seconds


The combine step doesn't need an LLM. It's a normal f-string.

In [19]:
report = f"""**Sentiment:** {sentiment.final_output}

**Issues:**
{issues.final_output}

**Suggested reply:**
{reply.final_output}"""

display(Markdown(report))

**Sentiment:** Mixed

**Issues:**
* Expired milk
* Coupon failed to apply at checkout
* 40-minute wait for support chat

**Suggested reply:**
We’re truly sorry for the expired milk, the checkout glitch, and the frustrating wait to reach our team. 

Please DM us your order ID so we can immediately refund the milk and make sure your coupon is applied.

3 LLM calls in roughly the time of 1.

Plain Python steps like the combine are faster, cheaper and never make things up. Use an LLM only where you need one.

**Use it when:** you have independent checks or views of the same input. Also for **voting**: ask 3 times, take the majority answer.

**Try it:** run the 3 agents one after another (3 separate `await Runner.run(...)` lines) and compare the time.

## 7. Pattern 4: ReAct Agent

So far, **we** decided every step (except the router's one choice). Now we let the **LLM decide**.

**ReAct** = **Re**ason + **Act**. The LLM works in a loop:

```
Think:    "I need the rupee to yen rate"
Act:      call convert_currency(...)
Observe:  read the result
Think:    "Now I need next week's weather in Tokyo"
Act:      call get_weather(...)
...       until it has enough to answer
```

Good news: **every SDK agent with tools is already a ReAct agent.** `Runner` runs this loop for you.

First, 2 tools. Both give Gemini something it **can't know on its own**: today's exchange rate and next week's weather. Both APIs are free, no key needed.

- `convert_currency` uses [Frankfurter](https://frankfurter.dev), a free exchange-rate API.
- `get_weather` uses [Open-Meteo](https://open-meteo.com), a free weather API. It finds the city's location, then gets a 7-day forecast.

Put `@function_tool` on top of any Python function with type hints and a docstring. Same idea as tools in Part 1: the docstring tells the LLM what the tool does.

In [20]:
import requests
from agents import function_tool


@function_tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert money between currencies at today's exchange rate. Use codes like USD, INR, EUR, JPY."""
    print(f"Tool called! Converting {amount} {from_currency} to {to_currency}")

    url = f"https://api.frankfurter.dev/v1/latest?amount={amount}&from={from_currency}&to={to_currency}"
    data = requests.get(url, timeout=15).json()
    return f"{amount} {from_currency} = {data['rates'][to_currency]} {to_currency}"


@function_tool
def get_weather(city: str) -> str:
    """Get the weather forecast for the next 7 days in a city: daily low and high temperature, and rain."""
    print("Tool called! Getting the weather for", city)

    # Step 1: Find the city's location (latitude and longitude).
    url = "https://geocoding-api.open-meteo.com/v1/search"
    places = requests.get(url, params={"name": city, "count": 1}, timeout=15).json()
    if not places.get("results"):
        return f"City not found: {city}"
    place = places["results"][0]

    # Step 2: Get the daily forecast for that location.
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": place["latitude"],
        "longitude": place["longitude"],
        "daily": "temperature_2m_min,temperature_2m_max,precipitation_sum",
        "timezone": "auto",
    }
    daily = requests.get(url, params=params, timeout=15).json()["daily"]

    # Step 3: One line per day.
    lines = []
    for day, low, high, rain in zip(daily["time"], daily["temperature_2m_min"], daily["temperature_2m_max"], daily["precipitation_sum"]):
        lines.append(f"{day}: {low}-{high}°C, rain {rain} mm")
    return "\n".join(lines)

In [21]:
travel_agent = Agent(
    name="Travel Assistant",
    model=MODEL,
    tools=[convert_currency, get_weather],
    instructions="You are a helpful travel assistant for FDEs visiting clients. Use tools for live rates and weather. Answer in markdown.",
)

question = "I'm visiting a client in Tokyo next week with 50,000 rupees. How many yen is that, and what should I pack for the weather?"

result = await Runner.run(travel_agent, question)

display(Markdown(result.final_output))

Tool called! Getting the weather forTool called! Converting 50000.0 INR to JPY
 Tokyo


### 💴 Currency Conversion

**₹50,000 INR** is approximately **¥85,592 JPY** at current rates.

---

### ⛅ Tokyo 7-Day Weather Forecast

| Date | Low / High Temp | Rain |
| :--- | :--- | :--- |
| **Day 1** | 4.6°C / 13.0°C | None |
| **Day 2** | 4.1°C / 13.0°C | None |
| **Day 3** | 7.1°C / 18.1°C | Light to moderate rain (6.3 mm) |
| **Day 4** | 5.2°C / 12.0°C | Minimal rain (0.5 mm) |
| **Day 5** | 3.1°C / 13.0°C | None |
| **Day 6** | 5.6°C / 16.2°C | None |
| **Day 7** | 7.5°C / 14.9°C | Rain likely (13.8 mm) |

---

### 🧳 Packing Recommendations

* **Warm Outerwear:** Mornings and evenings will be chilly (dropping to around 3°C–5°C). Bring a medium-weight coat, overcoat, or a warm jacket suitable for client meetings.
* **Layering Pieces:** Long-sleeve shirts, knitwear, cardigans, or blazers to adapt easily between heated indoor offices/trains and colder streets.
* **Rain Protection:** A compact, sturdy umbrella and a water-resistant jacket or trench coat.
* **Footwear:** Comfortable, smart-casual or formal walking shoes (with some water resistance), as client visits in Tokyo usually involve significant walking and transit.
* **Accessories:** A scarf and lightweight gloves are helpful if you tend to feel the cold, especially when commuting early in the morning or late at night.

### See which tools it chose

`result.new_items` lists everything that happened in the loop. We keep only the tool calls.

from agents import ToolCallItem

for item in result.new_items:
    if isinstance(item, ToolCallItem):
        print(item.raw_item.name, item.raw_item.arguments)

In [22]:
from agents import ToolCallItem

for item in result.new_items:
    if isinstance(item, ToolCallItem):
        print(item.raw_item.name, item.raw_item.arguments)

convert_currency {"from_currency": "INR", "amount": 50000, "to_currency": "JPY"}
get_weather {"city": "Tokyo"}


We never wrote the order. The LLM chose which tools, with which inputs, and when to stop.

**Safety net:** `Runner` stops after **10 turns** by default. You can change it with `Runner.run(agent, question, max_turns=5)`, so a confused agent can't loop forever.

**Use it when:** you can't know the steps in advance. Research, debugging, open questions.

**Watch out:** agents are less predictable and cost more (many LLM calls). For an FDE, that means **always test with real client examples** before going live

## 8. Pattern 5: Orchestrator–Worker

**The idea:** a lead LLM (the **orchestrator**) looks at the task, splits it into pieces, and hands each piece to the right **worker**. Then it combines their work.

```
                 +-->  Problem Analyst     --+
task --> leader  +-->  Solution Architect  --+-->  leader combines  -->  answer
                 +-->  Project Planner     --+
```

How is this different from the router?

- **Router:** picks **one** specialist, who answers
- **Orchestrator:** gives **different tasks** to **several** workers, then combines

In the SDK, you turn each worker into a tool with `agent.as_tool()`. The leader is a normal agent whose tools are other agents.

**FreshKart task:** write a proposal for their AI support bot.

In [23]:
problem_analyst = Agent(
    name="Problem Analyst",
    model=MODEL,
    instructions="You explain the client's problem and what it costs them. Write in plain business language. Max 80 words.",
)

solution_architect = Agent(
    name="Solution Architect",
    model=MODEL,
    instructions="You design the technical solution. Describe it simply, with the main parts. Max 80 words.",
)

project_planner = Agent(
    name="Project Planner",
    model=MODEL,
    instructions="You plan the timeline, team and rough cost. Give a short timeline in phases. Max 80 words.",
)

Each as_tool() needs a tool_name and a tool_description. The description is what the leader reads to decide who does what.

In [24]:
proposal_lead = Agent(
    name="Proposal Lead",
    model=MODEL,
    instructions=(
        "You lead a proposal team at an AI consulting firm. "
        "Give each worker the part of the proposal that matches their job. "
        "Combine their work into one proposal with a ## heading per section."
    ),
    tools=[
        problem_analyst.as_tool(
            tool_name="problem_analyst",
            tool_description="Explains the client's problem and what it costs them",
        ),
        solution_architect.as_tool(
            tool_name="solution_architect",
            tool_description="Designs the technical solution",
        ),
        project_planner.as_tool(
            tool_name="project_planner",
            tool_description="Plans the timeline, team and rough cost",
        ),
    ],
)

In [25]:
task = "Write a proposal for FreshKart: an AI bot that answers 'where is my order' support tickets automatically and passes the rest to humans."

result = await Runner.run(proposal_lead, task)

display(Markdown(result.final_output))

# AI-Powered Support Automation Proposal for FreshKart

## Problem Analysis & Business Impact
FreshKart’s support team is overwhelmed by repetitive "Where is my order?" (WISMO) inquiries, creating support backlogs and slow response times. 

This operational bottleneck inflates support expenditures by forcing paid agents to manually answer routine tracking questions instead of resolving high-touch, complex customer issues. More critically, delivery anxiety frustrates hungry shoppers, driving them directly to competitors. The true cost to FreshKart is twofold: bloated customer service overhead combined with lost repeat sales and eroded customer lifetime value in a fiercely competitive on-demand delivery market.

---

## Technical Solution Architecture
To eliminate manual overhead while ensuring a high standard of customer satisfaction, we propose an intelligent tier-1 automation pipeline composed of four core modules:

1. **Conversational AI & NLU:** Accurately classifies incoming customer tickets and chat queries into actionable intents (identifying WISMO vs. non-WISMO requests).
2. **Integration Middleware:** Securely connects to FreshKart’s Order Management System (OMS) and third-party logistics APIs to retrieve real-time fulfillment and courier tracking statuses.
3. **Automated Response Engine:** Instantly resolves identified WISMO queries by generating precise, dynamic delivery updates with zero human intervention.
4. **Smart Escalation Hub:** Automatically triages non-WISMO and edge-case tickets to the appropriate human agents via your existing CRM/helpdesk, passing the full conversation transcript and customer context for an effortless handoff.

---

## Project Timeline, Team & Investment

### Core Team
* **1 Project Manager:** Overall delivery, sprint planning, and client communication
* **1 AI/NLP Engineer:** Intent classification modeling, prompt tuning, and NLU optimization
* **1 Integration & Backend Developer:** OMS, logistics, and CRM/ticketing API connectors
* **1 QA Engineer:** End-to-end testing, edge-case validation, and regression suites

### Phased Timeline (8 Weeks)
* **Phase 1: Foundation & Core MVP (Weeks 1–3):** Connect FreshKart OMS/CRM; train and validate WISMO intent models.
* **Phase 2: Handoff & Business Logic (Weeks 4–5):** Implement human-agent escalation workflows, ticketing integrations, and internal alpha testing.
* **Phase 3: Pilot & Refinement (Weeks 6–7):** Deploy a controlled customer beta, fine-tune edge cases, and perform load/QA testing.
* **Phase 4: Launch & Stabilization (Week 8):** Full production rollout, analytics dashboard setup, and operational monitoring.

### Estimated Investment
* **Rough Cost Estimate:** $40,000 – $55,000 (covers all phases from architecture setup to production deployment).

### Who did what?

Same trick as the ReAct agent: look at the tool calls in `new_items`. `ToolCallOutputItem` holds what each worker sent back.

In [26]:
from agents import ToolCallOutputItem

for item in result.new_items:
    if isinstance(item, ToolCallItem):
        print("===== Leader asked:", item.raw_item.name, "=====")
        print(item.raw_item.arguments)
    if isinstance(item, ToolCallOutputItem):
        print("----- Worker replied -----")
        print(item.output)
        print()

===== Leader asked: problem_analyst =====
{"input": "Client: FreshKart (e-commerce/grocery delivery). Problem: High volume of 'Where is my order?' (WISMO) customer support tickets, burdening human agents, increasing response times, and driving up operational support costs while degrading customer experience."}
===== Leader asked: solution_architect =====
{"input": "Client: FreshKart. Requirement: An AI-powered customer support bot that automates resolution for 'Where is my order?' (WISMO) queries by integrating with order management/logistics APIs, while intelligently routing all non-WISMO or complex queries to human support agents with context."}
===== Leader asked: project_planner =====
{"input": "Client: FreshKart. Project: AI WISMO Support Bot with human handoff. Scope: Timeline, team composition, milestones, and rough cost estimate for an MVP through production rollout."}
----- Worker replied -----
FreshKart’s support team is overwhelmed by repetitive "Where is my order?" inquirie

The leader is a **ReAct agent whose tools are other agents**. It "calls" a worker the same way the travel agent called `convert_currency`.

### Handoff vs Agent-as-tool

Both connect agents, but they're different:

| | `handoffs=[...]` (Router) | `agent.as_tool()` (Orchestrator) |
|---|---|---|
| Who answers the user? | The specialist **takes over** | The leader **stays in charge** |
| How many specialists? | Usually one | As many as needed |
| Good for | Routing a conversation | Splitting a task and combining |

**Use it when:** a task splits into parts, but you don't know the exact parts until you see the input.

## 9. Pattern 6: Reflection

**The idea:** one LLM **writes**, another LLM **checks**. If the check fails, the writer tries again with the feedback. Like a junior and a senior reviewing work.

```
writer  -->  checker  --(good)-->  done
   ^            |
   +--(not good, here's feedback)
```

**FreshKart task:** write a cold email to FreshKart's CEO. Our team has strict rules for these emails.

The checker uses **structured output** (from Part 1). In the SDK, you give the agent an `output_type`. Then `final_output` is a Python object, not text.

Just 2 agents and a normal Python `for` loop. **You don't need a framework feature for every pattern.**

In [27]:
from pydantic import BaseModel

RULES = """
- Under 70 words
- Mentions FreshKart by name
- Includes one number (a stat or a result)
- Ends with a question
- No words like "revolutionary", "cutting-edge" or "game-changer"
"""


class Review(BaseModel):
    is_good: bool
    feedback: str


writer = Agent(
    name="Writer",
    model=MODEL,
    instructions="You write short cold emails for an AI consulting firm. Return only the email.",
)

checker = Agent(
    name="Checker",
    model=MODEL,
    instructions="Check the email strictly against these rules. If any rule fails, explain what to fix." + RULES,
    output_type=Review,
)

In [28]:
task = "Write a cold email to the CEO of FreshKart, offering an AI bot that answers their support tickets."

prompt = task

for attempt in range(1, 4):    # max 3 tries, so the loop always ends
    print(f"Writer: attempt {attempt}")
    draft = (await Runner.run(writer, prompt)).final_output

    review = (await Runner.run(checker, draft)).final_output
    if review.is_good:
        print("Checker: good ✅")
        break

    print("Checker: not good ->", review.feedback)
    prompt = f"{task}\n\nYour last draft:\n{draft}\n\nFix these problems:\n{review.feedback}"

print()
print(draft)

Writer: attempt 1
Checker: not good -> The email fails multiple rules: 1. Word count: At around 74-79 words (including subject/signature), it is not under 70 words. 2. Number rule: It contains two numbers ('60%+' and '5-minute') instead of just one number representing a stat or result. Remove '5-minute'. 3. Ending: It ends with a sign-off/signature rather than directly ending with a question.
Writer: attempt 2
Checker: good ✅

Subject: AI support for FreshKart

Hi [First Name],

With FreshKart’s growth, your team is likely buried in repetitive tickets about delivery delays and missing items.

We build custom AI agents that automatically resolve 60% of frontline support tickets without human intervention, maintaining high CSAT.

Worth exploring how much support overhead we could take off your plate this week?


The writer never sees the rules. It only gets the checker's feedback, and fixes what it's told.

If the checker passed it on the first try, make the rules stricter (for example "under 40 words") and run again to see the loop. If all 3 tries fail, you still get the last draft, so a human can finish it.

Why does this work? Checking is easier than writing. An LLM that misses a rule while writing often spots it when it's only asked to check.

Use it when: there are clear rules for "good": word limits, required facts, code that must pass tests.

## 10. Pattern 7: Human in the Loop

**The idea:** some actions are too important to let AI do alone: emailing a client, issuing a refund, deleting data. So the agent **pauses**, asks a human, and continues based on the answer.

<img src="https://raw.githubusercontent.com/pratik-gond/FDE_images/main/human_in_the_loop.png" width="260">

**FreshKart task:** the agent replies to an angry customer. A human must approve the reply before it's sent.

In the SDK, you mark the risky tool with `@function_tool(needs_approval=True)`. The agent can **ask** to use it, but it won't run until a human says yes.

In [29]:
@function_tool(needs_approval=True)
def send_reply(message: str) -> str:
    """Send a reply message to the customer."""
    print("📧 Reply sent to the customer!")
    return "Reply sent"


support_agent = Agent(
    name="Support Agent",
    model=MODEL,
    tools=[send_reply],
    instructions=(
        "You are FreshKart's support agent. "
        "Write a short, polite reply to the customer and send it with send_reply. "
        "Never promise a refund amount. "
        "If a reply is rejected, rewrite it using the feedback and try sending again."
    ),
)

### Run it

- When the agent wants to use `send_reply`, the run **pauses**: `result.interruptions` lists the tool calls waiting for a human.
- Each interruption has the tool's `arguments` as JSON text. We read the draft `message` from it.
- `result.to_state()` gives the paused run's **state**. We `approve()` or `reject()` each interruption on it.
- `Runner.run(support_agent, state)` carries on from where it paused.

Type **yes** to send, or type what to change (for example "make it shorter" or "offer a 10% coupon").

In [31]:
import json

complaint = "My order came 2 hours late and the ice cream had melted. I want a full refund NOW."

result = await Runner.run(support_agent, complaint)

while result.interruptions:
    state = result.to_state()

    for interruption in result.interruptions:
        draft = json.loads(interruption.arguments)["message"]
        print("\n--- DRAFT ---\n" + draft + "\n-------------")

        answer = input("Type 'yes' to send, or write what to change: ")
        if answer.strip().lower() == "yes":
            state.approve(interruption)
        else:
            state.reject(interruption, rejection_message="Not sent. The manager wants this changed: " + answer)

    result = await Runner.run(support_agent, state)

print(result.final_output)


--- DRAFT ---
We sincerely apologize for the delay and for the condition of your ice cream. We understand your frustration. Could you please share your order ID so we can look into this right away and assist you with a resolution?
-------------
Type 'yes' to send, or write what to change: yes
📧 Reply sent to the customer!
I have sent a reply to the customer apologizing for the issue and requesting their order details to look into a resolution.


When you reject, your note goes back to the agent as the tool's result. It rewrites the reply and asks again, until you say yes.

**In a real app** the pause could last hours (a manager approves on Slack, say). Save the paused run with `saved = state.to_string()`, and later load it with `state = await RunState.from_string(support_agent, saved)` and continue.

**Use it when:** an action is costly, can't be undone, or faces a customer. In FDE work, this is often what makes a client say yes to going live.

## 11. Which Pattern When?

| Pattern | In OpenAI Agents SDK | Use it when | FreshKart example |
|---|---|---|---|
| Prompt chaining | `Runner.run` one after another | Fixed steps, one after another | Meeting notes → email |
| Router | `handoffs=[...]` | Different inputs need different handling | Ticket → right team |
| Parallel calls | `asyncio.gather` | Independent jobs on the same input | Review → sentiment + issues + reply |
| ReAct agent | `Agent` with `tools` | Steps aren't known in advance | Travel questions with live tools |
| Orchestrator–worker | `agent.as_tool()` | Task splits into parts that need different experts | Proposal from 3 specialists |
| Reflection | `output_type` + a `for` loop | Clear rules for "good" | Cold email that follows the rules |
| Human in the loop | `@function_tool(needs_approval=True)` | Risky or customer-facing actions | Approve reply before sending |

**These patterns combine.** A real FreshKart bot is a **router** that sends "where is my order" tickets to a **ReAct agent**, with a **human in the loop** before any refund. Let's build exactly that.